# B2-019 — Session 4: Attention Module and Tiny Training

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Module contract

A minimal module owns four linear maps: query, key, value, and output. It validates rank 3, shared batch/sequence shapes, divisibility by heads, and Boolean mask broadcastability.

**Checkpoint 1A.** Which projection widths must agree before score multiplication?

**Checkpoint 1B.** Why store the scale as $d_h^{-1/2}$?

In [ ]:
import torch
from torch import nn
SEED = 20260808
torch.manual_seed(SEED)
DEVICE = torch.device('cpu')
assert DEVICE.type == 'cpu'

## 2. Forward pass and mask

Project, split heads, compute scaled scores, add the mask before softmax, multiply by values, concatenate, then apply the output projection. Test exact shapes and certify forbidden weights are zero.

**Worked example 1.** An identity-projection single-head module must reproduce the NumPy computation from Session 2.

**Checkpoint 2A.** Which dimension receives softmax?

**Checkpoint 2B.** What mask dtype does this unit require?

In [ ]:
def split_heads(x, heads):
    batch, length, width = x.shape
    assert width % heads == 0
    return x.reshape(batch, length, heads, width // heads).transpose(1, 2)
probe = torch.zeros(2, 4, 8)
assert split_heads(probe, 2).shape == (2, 2, 4, 4)

## 3. Seeded causal prediction task

Inputs are a pinned one-hot tensor with positional signals added numerically. At token $i$, the target is the next token class. A causal mask prevents the model from reading future targets.

**Checkpoint 3A.** Why shift inputs and targets by one position?

**Checkpoint 3B.** Which rows contribute to cross-entropy?

In [ ]:
logits = torch.zeros(2, 3, requires_grad=True)
targets = torch.tensor([0, 2])
loss = nn.functional.cross_entropy(logits, targets)
loss.backward()
assert torch.isclose(loss.detach(), torch.log(torch.tensor(3.0)), atol=1e-7, rtol=1e-7)
assert logits.grad is not None

## 4. Training loop

Set seed 20260808, instantiate on CPU, compute logits, call cross-entropy on flattened logits and integer targets, then run zero_grad, backward, and optimizer step. Record a loss trace and deterministic probe logits.

**Worked example 2.** A uniform three-class predictor has loss $\log 3$ per row.

**Checkpoint 4A.** Should softmax be called before cross-entropy?

**Checkpoint 4B.** What makes the experiment reproducible?

## 5. Common pitfalls

Broken: build a float mask but interpret True as forbidden in one place and allowed in another. Fix: name it `allowed` and test corner cells. Broken: detach logits before loss. Fix: keep the computation graph through the scalar loss.

**Exam connections.** From-scratch tasks grade scale, mask placement, shape contract, and actual gradient-based parameter updates separately.

**Going deeper.** Session 5 wraps attention with residual and feed-forward sublayers.

Checkpoint answers: 1A query/key head width; 1B stable explicit scale; 2A key axis; 2B Boolean; 3A next-step supervision; 3B nonpadding prediction positions; 4A no; 4B fixed inputs, seed, CPU, APIs, and tolerances.